In [ ]:
from transformers import ViTForImageClassification,ViTImageProcessor,Trainer, TrainingArguments
from torch.utils.data import DataLoader, TensorDataset
from datasets import load_dataset
from torchvision.transforms import Compose, Resize, ToTensor, Normalize
from sklearn.model_selection import KFold
from PIL import Image
from tqdm import tqdm
import itertools
import random
import torchvision
import torch
import cv2
import json
import pandas as pd
import numpy as np
import glob
import matplotlib.pyplot as plt
from sklearn.metrics import confusion_matrix,classification_report
import numpy as np
from sklearn.metrics import accuracy_score
import seaborn as sns
from torchsummary import summary
import torch

In [ ]:
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

# **Descarga componentes ViT**

In [ ]:
model = ViTForImageClassification.from_pretrained('google/vit-base-patch16-224-in21k', num_labels=2)
model.to(device)

/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:124: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


config.json:   0%|          | 0.00/502 [00:00<?, ?B/s]

model.safetensors:   0%|          | 0.00/346M [00:00<?, ?B/s]

Loading weights:   0%|          | 0/6 [00:00<?, ?it/s]

[transformers] ViTForImageClassification LOAD REPORT from: google/vit-base-patch16-224-in21k
Key                                                     | Status     | 
--------------------------------------------------------+------------+-
encoder.layer.{0...11}.attention.attention.key.weight   | UNEXPECTED | 
encoder.layer.{0...11}.attention.output.dense.weight    | UNEXPECTED | 
encoder.layer.{0...11}.intermediate.dense.weight        | UNEXPECTED | 
pooler.dense.weight                                     | UNEXPECTED | 
encoder.layer.{0...11}.attention.attention.value.weight | UNEXPECTED | 
encoder.layer.{0...11}.output.dense.bias                | UNEXPECTED | 
encoder.layer.{0...11}.layernorm_before.weight          | UNEXPECTED | 
encoder.layer.{0...11}.attention.output.dense.bias      | UNEXPECTED | 
encoder.layer.{0...11}.attention.attention.query.weight | UNEXPECTED | 
encoder.layer.{0...11}.attention.attention.value.bias   | UNEXPECTED | 
encoder.layer.{0...11}.layernorm_after.weig

ViTForImageClassification(
  (vit): ViTModel(
    (embeddings): ViTEmbeddings(
      (patch_embeddings): ViTPatchEmbeddings(
        (projection): Conv2d(3, 768, kernel_size=(16, 16), stride=(16, 16))
      )
      (dropout): Dropout(p=0.0, inplace=False)
    )
    (layers): ModuleList(
      (0-11): 12 x ViTLayer(
        (attention): ViTAttention(
          (q_proj): Linear(in_features=768, out_features=768, bias=True)
          (k_proj): Linear(in_features=768, out_features=768, bias=True)
          (v_proj): Linear(in_features=768, out_features=768, bias=True)
          (o_proj): Linear(in_features=768, out_features=768, bias=True)
        )
        (layernorm_before): LayerNorm((768,), eps=1e-12, elementwise_affine=True)
        (layernorm_after): LayerNorm((768,), eps=1e-12, elementwise_affine=True)
        (mlp): ViTMLP(
          (activation_fn): GELUActivation()
          (fc1): Linear(in_features=768, out_features=3072, bias=True)
          (fc2): Linear(in_features=3072, out

In [ ]:
feature_extractor = ViTImageProcessor.from_pretrained('google/vit-base-patch16-224-in21k')

preprocessor_config.json:   0%|          | 0.00/160 [00:00<?, ?B/s]

# **Clase ImageDataset**


In [ ]:
class ImageDataset(torch.utils.data.Dataset):
    def __init__(self, paths, labels):
        self.images = paths
        self.labels = labels

    def __len__(self):
      return len(self.images)

    def __getitem__(self, idx):
      path = self.images[idx]
      image = Image.open(self.images[idx])#.convert('L') # Convert to grayscale
      transform = torchvision.transforms.ToTensor()
      image_tensor = transform(image)
      label = self.labels[idx]
      return image_tensor, torch.tensor(label, dtype=torch.long), path

# **Preparación de datos**


In [ ]:
random.seed(3434)
np.random.seed(3434)
torch.manual_seed(3434)
torch.cuda.manual_seed_all(3434)
t_type = 'Meander'
path_H = f"/content/drive/MyDrive/{t_type}Control_out_resize/"
path_S = f"/content/drive/MyDrive/{t_type}Patients_out_resize/"
path_schema = f"/content/drive/MyDrive/{t_type}_id_train_test_60_percent.json"


Funciones para cargar los datos de los ids definidos en las variables anteriores.

In [ ]:
def load_ids_data(path):
    """
    @path : ruta del fichero json
    @return : diccionario con los datos del fichero
    """
    with open(path, "r") as f:
      data = json.load(f)
    return data


def search_data(ids, status):
    """
    @ids : lista con los ids de los pacientes
    @status : 0 -> Healthy, 1 -> Sick
    @return : lista de la ruta de las imagenes de los pacientes
    """
    list_data = []
    path = path_H if status == 0 else path_S
    for i in glob.glob(path+"*_TH.jpg"):
      user = i.split("/")[-1].split("-")[0]
      if user in ids:
        list_data.append(i)
    #
    labels = list(itertools.repeat(status, len(list_data)))
    #
    return list_data, labels


def createDataset(ids):
    """
    @ids : json con los ids a utilizar para la red
    @return : objeto de tipo ImageDataset
    """
    data_h, labels_h = search_data(ids['healthy'], 0)
    data_s, labels_s = search_data(ids['sick'], 1)
    data_all = data_h + data_s
    labels_all = labels_h + labels_s
    #
    return ImageDataset(data_all, labels_all)


In [ ]:
ids_process = load_ids_data(path_schema)
trainDataset = createDataset(ids_process['id_train'])
testDataset = createDataset(ids_process['id_test'])


In [ ]:
# The `AttributeError: 'ImageDataset' object has no attribute 'with_transform'`
# occurs because `ImageDataset` is a custom PyTorch Dataset, not a HuggingFace `datasets` library object.
# PyTorch `Dataset` objects do not have a `with_transform` method by default.
# Also, the ImageDataset currently returns grayscale (1-channel) tensors, while ViT expects 3-channel input.

# Redefine the transformations to work on the tensor output by ImageDataset
# and handle the channel replication.
image_transforms = Compose([
    # Resize the image tensor to the expected ViT input size
    Resize((224, 224)),
    # Convert 1-channel grayscale tensor to 3-channel by repeating
    lambda x: x.repeat(3, 1, 1) if x.shape[0] == 1 else x,
    # Normalize the tensor using ViT specific mean and std values
    Normalize(mean=feature_extractor.image_mean, std=feature_extractor.image_std)
])

class CustomTransformedDataset(torch.utils.data.Dataset):
    def __init__(self, original_dataset, transform_fn):
        self.original_dataset = original_dataset
        self.transform_fn = transform_fn

    def __len__(self):
        return len(self.original_dataset)

    def __getitem__(self, idx):
        # ImageDataset returns image_tensor (1, H, W), label, path
        image_tensor, label, path = self.original_dataset[idx]
        transformed_image_tensor = self.transform_fn(image_tensor)
        # Return as a dictionary, compatible with HuggingFace Trainer input format
        return {'pixel_values': transformed_image_tensor, 'labels': label}

# Apply the transformation by creating new datasets that wrap the original ones
prepared_dataset_train = CustomTransformedDataset(trainDataset, image_transforms)
prepared_dataset_test = CustomTransformedDataset(testDataset, image_transforms)

In [ ]:
trainloader = torch.utils.data.DataLoader(prepared_dataset_train,
                                                  shuffle=True, batch_size=32)
testloader = torch.utils.data.DataLoader(prepared_dataset_test,batch_size=32)

# **Entrenamiento de la red**

In [ ]:
# Define training arguments
training_args = TrainingArguments(
    output_dir='./results',             # output directory
    per_device_train_batch_size=32,     # batch size for training
    per_device_eval_batch_size=32,      # batch size for evaluation
    num_train_epochs=60,                 # number of training epochs
    eval_strategy="epoch",        # evaluate after each epoch
    save_strategy="epoch",              # save checkpoint after each epoch
    logging_dir='./logs',               # logging directory
    logging_steps=10,
    load_best_model_at_end=True,        # Load best model after training
)

# Define a simple evaluation metric
from sklearn.metrics import accuracy_score
def compute_metrics(pred):
    labels = pred.label_ids
    preds = pred.predictions.argmax(-1)
    acc = accuracy_score(labels, preds)
    return {"accuracy": acc}

# Define Trainer with model, data, and training arguments
trainer = Trainer(
    model=model,
    args=training_args,
    train_dataset=prepared_dataset_train,
    eval_dataset=prepared_dataset_test,
    compute_metrics=compute_metrics
)
# Fine-tune the model
trainer.train()

[transformers] `logging_dir` is deprecated and will be removed in v5.2. Please set `TENSORBOARD_LOGGING_DIR` instead.


Epoch,Training Loss,Validation Loss,Accuracy
1,No log,0.354619,0.892361
2,No log,0.553968,0.774306
3,No log,0.595111,0.725694
4,0.545213,0.364430,0.861111
5,0.545213,0.650259,0.701389
6,0.545213,0.455912,0.809028
7,0.284029,0.970635,0.690972
8,0.284029,1.111455,0.680556
9,0.284029,0.546355,0.829861
10,0.135522,1.080709,0.677083


Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

...

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

In [ ]:
# Define training arguments  --> Meanders
training_args = TrainingArguments(
    output_dir='./results',             # output directory
    per_device_train_batch_size=32,     # batch size for training
    per_device_eval_batch_size=32,      # batch size for evaluation
    num_train_epochs=60,                 # number of training epochs
    eval_strategy="epoch",        # evaluate after each epoch
    save_strategy="epoch",              # save checkpoint after each epoch
    logging_dir='./logs',               # logging directory
    logging_steps=10,
    load_best_model_at_end=True,        # Load best model after training
)

# Define a simple evaluation metric
from sklearn.metrics import accuracy_score
def compute_metrics(pred):
    labels = pred.label_ids
    preds = pred.predictions.argmax(-1)
    acc = accuracy_score(labels, preds)
    return {"accuracy": acc}

# Define Trainer with model, data, and training arguments
trainer = Trainer(
    model=model,
    args=training_args,
    train_dataset=prepared_dataset_train,
    eval_dataset=prepared_dataset_test,
    compute_metrics=compute_metrics
)
# Fine-tune the model
trainer.train()

[transformers] `logging_dir` is deprecated and will be removed in v5.2. Please set `TENSORBOARD_LOGGING_DIR` instead.


Epoch,Training Loss,Validation Loss,Accuracy
1,No log,1.194015,0.111111
2,No log,0.571415,0.888889
3,No log,0.734102,0.111111
4,0.779862,0.849069,0.111111
5,0.779862,0.757325,0.111111
6,0.779862,0.612922,0.888889
7,0.690647,0.607737,0.892361
8,0.690647,0.710917,0.340278
9,0.690647,0.759073,0.215278
10,0.660211,0.634481,0.690972


Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

...

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

TrainOutput(global_step=180, training_loss=0.2680282566282484, metrics={'train_runtime': 3003.577, 'train_samples_per_second': 1.598, 'train_steps_per_second': 0.06, 'total_flos': 3.719615501500416e+17, 'train_loss': 0.2680282566282484, 'epoch': 60.0})